### **Lấy và làm sạch Dữ liệu AQI và Weather từ AQI**

![Ảnh](./images/pam1_JLLG.png)

#### **I. Import các thư viện**

In [1]:
import requests
import json
import pandas as pd
import numpy as np
import datetime
from datetime import datetime, timedelta
from time import sleep
%matplotlib inline      

#### **II. Lấy dữ liệu từ API và tạo Dataframe từ data lấy được**
Trong dự án này, dữ liệu chất lượng không khí được thu thập thông qua [Weatherbit API](https://www.weatherbit.io/), một nền tảng tổng hợp dữ liệu khí tượng – môi trường toàn cầu.Weatherbit không trực tiếp đặt cảm biến tại từng vị trí địa lý, mà tổng hợp dữ liệu từ nhiều nguồn khác nhau, bao gồm:
- Các trạm quan trắc mặt đất (ground-based stations),

- Dữ liệu vệ tinh khí tượng (NASA MODIS, TROPOMI, MOPITT, v.v.),

- Và các mô hình khí tượng – hóa học (numerical models) như WRF-Chem, CAMS, GEOS-Chem.

Dữ liệu sau khi được thu thập sẽ được hiệu chỉnh sai số, nội suy không gian và chuẩn hóa theo từng ô lưới (grid) có kích thước khoảng 10 km × 10 km.
Điều này có nghĩa là các khu vực nằm trong cùng một ô lưới (ví dụ các quận nội thành Hà Nội nằm gần nhau) sẽ nhận được cùng một giá trị AQI và nồng độ các chất ô nhiễm (PM₂.₅, PM₁₀, CO, NO₂, SO₂, O₃).

Vì vậy, để tránh hiện tượng dữ liệu bị trùng lặp (nhân bản) giữa các quận lân cận, dự án lựa chọn tọa độ trung tâm của quận Hoàn Kiếm (21.0285°N, 105.8542°E) làm điểm đại diện cho khu vực nội thành Hà Nội. Hoàn Kiếm là khu vực trung tâm thủ đô, có mật độ dân cư cao, nhiều hoạt động giao thông và thương mại, do đó phản ánh tương đối chính xác chất lượng không khí trung bình của toàn khu vực đô thị Hà Nội.


In [10]:
# API_KEY = "d0abdba555a24c308b658ff1a9af5267"
# API_KEY = "23c63a63d23b4e18bc2902d841b53ce2"
API_KEY = "7dbe16d6d1c54c10bc1f827a407861a0"
LAT, LON = 21.0285, 105.8542


# Khoang thoi gian theo gio dia phuong: 00:00 13/01/2022 -> 23:00 31/08/2026
start_date = datetime(2022, 1, 13)
end_date = datetime(2026, 9, 1)  # moc ket thuc khong bao gom

urls_air = []
urls_wea = []
current = start_date
while current < end_date:
    next_month = (current.replace(day=28) + timedelta(days=4)).replace(day=1)
    chunk_end = min(next_month, end_date)
    start_str = current.strftime('%Y-%m-%d')
    end_str = chunk_end.strftime('%Y-%m-%d')

    url_air = f"https://api.weatherbit.io/v2.0/history/airquality?lat={LAT}&lon={LON}&start_date={start_str}&end_date={end_str}&tz=local&key={API_KEY}"
    url_wea = f"https://api.weatherbit.io/v2.0/history/hourly?lat={LAT}&lon={LON}&start_date={start_str}&end_date={end_str}&tz=local&key={API_KEY}"
    urls_air.append(url_air)
    urls_wea.append(url_wea)
    current = chunk_end

print(f" Tạo {len(urls_air)} URLs ({urls_air[0]} → {urls_air[-1]})")
print(f" Tạo {len(urls_wea)} URLs ({urls_wea[0]} → {urls_wea[-1]})")


 Tạo 56 URLs (https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2022-01-13&end_date=2022-02-01&tz=local&key=7dbe16d6d1c54c10bc1f827a407861a0 → https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2026-08-01&end_date=2026-09-01&tz=local&key=7dbe16d6d1c54c10bc1f827a407861a0)
 Tạo 56 URLs (https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2022-01-13&end_date=2022-02-01&tz=local&key=7dbe16d6d1c54c10bc1f827a407861a0 → https://api.weatherbit.io/v2.0/history/hourly?lat=21.0285&lon=105.8542&start_date=2026-08-01&end_date=2026-09-01&tz=local&key=7dbe16d6d1c54c10bc1f827a407861a0)


##### **2.1 AIR QUALITY**

In [6]:
results_air = []
for i, url in enumerate(urls_air):
    print(f'Lấy dữ liệu từ URL {i}/{len(urls_air)} : {url}')
    try:
        renponse = requests.get(url, timeout=30)
        renponse.raise_for_status()
        data = json.loads(renponse.text)
        results_air.append(data)
        sleep(1.2)

    except Exception as e:
        print(f"Lỗi khi lấy dữ liệu {i} : {e}")

print(f"\n Hoàn tất tải {len(results_air)} ")

Lấy dữ liệu từ URL 0/56 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2022-01-13&end_date=2022-02-01&tz=local&key=7dbe16d6d1c54c10bc1f827a407861a0
Lấy dữ liệu từ URL 1/56 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2022-02-01&end_date=2022-03-01&tz=local&key=7dbe16d6d1c54c10bc1f827a407861a0
Lấy dữ liệu từ URL 2/56 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2022-03-01&end_date=2022-04-01&tz=local&key=7dbe16d6d1c54c10bc1f827a407861a0
Lấy dữ liệu từ URL 3/56 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2022-04-01&end_date=2022-05-01&tz=local&key=7dbe16d6d1c54c10bc1f827a407861a0
Lấy dữ liệu từ URL 4/56 : https://api.weatherbit.io/v2.0/history/airquality?lat=21.0285&lon=105.8542&start_date=2022-05-01&end_date=2022-06-01&tz=local&key=7dbe16d6d1c54c10bc1f827a407861a0
Lấy dữ liệu từ URL 5/56 : https://api.weatherbit.io/v2.

In [ ]:
# Chỉ tải lại tháng cuối
response = requests.get(urls_air[-1], timeout=60)
response.raise_for_status()

results_air[-1] = response.json()

print("Đã cập nhật request cuối")
print("Số phần dữ liệu:", len(results_air))

In [ ]:
results_air[0]['city_name']

In [ ]:
results_air[0]['data'][0]

In [9]:
joined_data = []
for res in results_air:
    if 'data' in res:
        joined_data.extend(res['data'])

joined_results = {
    'city_name': results_air[0]['city_name'],
    'country_code': results_air[0]['country_code'],
    'lat': results_air[0]['lat'],
    'lon': results_air[0]['lon'],
    'timezone': results_air[0]['timezone'],
    'data': joined_data
}


In [10]:
joined_results['data'][0]

{'aqi': 100,
 'co': 229.5,
 'datetime': '2022-01-31:17',
 'no2': 10.7,
 'o3': 68.7,
 'pm10': 49.7,
 'pm25': 35,
 'so2': 37.3,
 'timestamp_local': '2022-02-01T00:00:00',
 'timestamp_utc': '2022-01-31T17:00:00',
 'ts': 1643648400}

In [11]:
joined_results['data'][-1]

{'aqi': 66,
 'co': 160.5,
 'datetime': '2026-07-31:17',
 'no2': 33,
 'o3': 15.5,
 'pm10': 25,
 'pm25': 19.33,
 'so2': 25,
 'timestamp_local': '2026-08-01T00:00:00',
 'timestamp_utc': '2026-07-31T17:00:00',
 'ts': 1785517200}

In [12]:

df = pd.DataFrame(joined_results)

df.columns = ['City', 'Country code', 'Lat', 'Lon', 'timezone', 'Data']
df[['AQI', 'CO', 'Date Time', 'NO2', 'O3', 'PM10', 'PM25', 'SO2', 'Local Time', 'UTC Time', 'TS']] = pd.DataFrame(df['Data'].tolist())
df.drop(columns=['Data', 'Lat', 'Lon', 'TS', 'Date Time'], inplace=True)
df['Local Time'] = pd.to_datetime(df['Local Time'])
df = df[(df['Local Time'] >= '2022-01-13 00:00:00') &
        (df['Local Time'] < '2026-09-01 00:00:00')]
df = (df.sort_values('Local Time')
        .drop_duplicates(subset=['Local Time'], keep='last')
        .set_index('Local Time'))
df

,City,Country code,timezone,AQI,CO,NO2,O3,PM10,PM25,SO2,UTC Time
Local Time,,,,,,,,,,,
2022-01-13 00:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,180.0,381.3,16.7,86.7,109.3,77.67,54.7,2022-01-12T17:00:00
2022-01-13 01:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,181.0,389.8,17.0,87.0,111.0,79.00,59.0,2022-01-12T18:00:00
2022-01-13 02:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,178.0,385.0,16.0,87.3,107.3,76.33,58.7,2022-01-12T19:00:00
2022-01-13 03:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,174.0,380.1,15.0,87.7,103.7,73.67,58.3,2022-01-12T20:00:00
2022-01-13 04:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,171.0,375.3,14.0,88.0,100.0,71.00,58.0,2022-01-12T21:00:00
...,...,...,...,...,...,...,...,...,...,...,...
2026-08-30 20:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,153.0,119.1,104.0,41.7,78.0,58.00,3.0,2026-08-30T13:00:00
2026-08-30 21:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,118.0,120.2,103.3,28.6,52.5,42.00,3.7,2026-08-30T14:00:00
2026-08-30 22:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,158.0,121.4,102.0,40.2,94.0,61.50,4.0,2026-08-30T15:00:00


In [ ]:
df.to_csv('air_quality_data_1.csv', index=True)

In [14]:
df = pd.read_csv('air_quality_data_1.csv')
df.shape

(40595, 12)

##### **2.2 WEATHER**

In [4]:
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

weather_cache_dir = project_root / 'data' / 'raw' / 'weatherbit_cache_20220113_20260831'
weather_cache_dir.mkdir(parents=True, exist_ok=True)

results_wea = []
for i, url in enumerate(urls_wea):
    cache_file = weather_cache_dir / f'weather_{i:03d}.json'

    # Neu thang nay da tai thanh cong thi doc lai, khong ton request
    if cache_file.exists():
        with cache_file.open('r', encoding='utf-8') as f:
            results_wea.append(json.load(f))
        print(f'Da co {i + 1}/{len(urls_wea)}, bo qua')
        continue

    print(f'Dang tai {i + 1}/{len(urls_wea)}')
    try:
        response = requests.get(url, timeout=60)
        if response.status_code == 429:
            print('Da het quota Weatherbit. Dung tai day va chay lai sau khi quota reset.')
            break
        response.raise_for_status()
        data = response.json()

        with cache_file.open('w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False)

        results_wea.append(data)
        sleep(1.2)

    except requests.RequestException as e:
        print(f'Loi khi tai request {i + 1}: {e}')
        print('Da dung. Chay lai cell nay de tiep tuc tu request chua tai.')
        break

print(f'Hoan tat {len(results_wea)}/{len(urls_wea)} request')
if len(results_wea) != len(urls_wea):
    print('CHUA DU DU LIEU: khong chay cac cell ghep va luu CSV ben duoi.')

Da co 1/56, bo qua
Da co 2/56, bo qua
Da co 3/56, bo qua
Da co 4/56, bo qua
Da co 5/56, bo qua
Da co 6/56, bo qua
Da co 7/56, bo qua
Da co 8/56, bo qua
Da co 9/56, bo qua
Da co 10/56, bo qua
Da co 11/56, bo qua
Da co 12/56, bo qua
Da co 13/56, bo qua
Da co 14/56, bo qua
Da co 15/56, bo qua
Da co 16/56, bo qua
Da co 17/56, bo qua
Da co 18/56, bo qua
Da co 19/56, bo qua
Da co 20/56, bo qua
Da co 21/56, bo qua
Da co 22/56, bo qua
Da co 23/56, bo qua
Da co 24/56, bo qua
Da co 25/56, bo qua
Da co 26/56, bo qua
Da co 27/56, bo qua
Da co 28/56, bo qua
Da co 29/56, bo qua
Da co 30/56, bo qua
Da co 31/56, bo qua
Da co 32/56, bo qua
Da co 33/56, bo qua
Da co 34/56, bo qua
Da co 35/56, bo qua
Da co 36/56, bo qua
Da co 37/56, bo qua
Da co 38/56, bo qua
Da co 39/56, bo qua
Da co 40/56, bo qua
Da co 41/56, bo qua
Da co 42/56, bo qua
Da co 43/56, bo qua
Da co 44/56, bo qua
Da co 45/56, bo qua
Da co 46/56, bo qua
Da co 47/56, bo qua
Da co 48/56, bo qua
Da co 49/56, bo qua
Da co 50/56, bo qua
Da co 51/

In [5]:
joined_data = []
for res in results_wea:
    if 'data' in res:
        joined_data.extend(res['data'])

joined_results_weather = {
    'city_name': results_wea[0]['city_name'],
    'country_code': results_wea[0]['country_code'],
    'lat': results_wea[0]['lat'],
    'lon': results_wea[0]['lon'],
    'timezone': results_wea[0]['timezone'],
    'data': joined_data
}

In [6]:

df_weather = pd.DataFrame(joined_results_weather)
df_weather.columns = ['City', 'Country code', 'Lat', 'Lon', 'timezone', 'Data']
d = pd.json_normalize(df_weather.pop('Data'))
selected_columns = [
    'clouds', 'precip', 'pres', 'slp', 'rh', 'temp', 'app_temp',
    'dewpt', 'uv', 'vis', 'wind_spd', 'wind_gust_spd', 'wind_dir',
    'solar_rad', 'timestamp_local', 'timestamp_utc'
]
rename_columns = {
    'clouds': 'Clouds',
    'precip': 'Precipitation',
    'pres': 'Pressure',
    'slp': 'Sea Level Pressure',
    'rh': 'Relative Humidity',
    'temp': 'Temperature',
    'app_temp': 'Apparent Temperature',
    'dewpt': 'Dew Point',
    'uv': 'UV Index',
    'vis': 'Visibility',
    'wind_spd': 'Wind Speed',
    'wind_gust_spd': 'Wind Gust Speed',
    'wind_dir': 'Wind Direction',
    'solar_rad': 'Solar Radiation',
    'timestamp_local': 'Local Time',
    'timestamp_utc': 'UTC Time'
}

# Chi chon cac cot Weatherbit thuc su tra ve de tranh KeyError
available_columns = [col for col in selected_columns if col in d.columns]
missing_columns = [col for col in selected_columns if col not in d.columns]
if missing_columns:
    print(f'Weatherbit khong tra ve cac cot: {missing_columns}')

keep = d[available_columns].rename(columns=rename_columns)
df_weather = pd.concat(
    [df_weather[['City', 'Country code', 'Lat', 'Lon', 'timezone']], keep],
    axis=1
)

# Chuan hoa, loc dung khoang thoi gian va loai bo ban ghi trung
df_weather['Local Time'] = pd.to_datetime(df_weather['Local Time'])
df_weather = df_weather[(df_weather['Local Time'] >= '2022-01-13 00:00:00') &
                        (df_weather['Local Time'] < '2026-09-01 00:00:00')]
df_weather = (df_weather.drop_duplicates(subset=['Local Time'])
                        .sort_values('Local Time')
                        .reset_index(drop=True))

expected_weather_hours = pd.date_range('2022-01-13 00:00:00',
                                       '2026-08-31 23:00:00', freq='h')
missing_weather_hours = expected_weather_hours.difference(
    pd.DatetimeIndex(df_weather['Local Time'])
)
print(f'So dong: {len(df_weather)}/40608 | So gio thieu: {len(missing_weather_hours)}')
print(f'Tu {df_weather["Local Time"].min()} den {df_weather["Local Time"].max()}')
df_weather.head()

So dong: 40608/40608 | So gio thieu: 0
Tu 2022-01-13 00:00:00 den 2026-08-31 23:00:00


,City,Country code,Lat,Lon,timezone,Clouds,Precipitation,Pressure,Sea Level Pressure,Relative Humidity,...,Apparent Temperature,Dew Point,UV Index,Visibility,Wind Speed,Wind Gust Speed,Wind Direction,Solar Radiation,Local Time,UTC Time
0,Hoàn Kiếm,VN,21.0285,105.8542,Asia/Ho_Chi_Minh,100,0.0,1019,1020,93,...,17.0,15.9,0.0,10.0,1.6,2.4,106,0,2022-01-13 00:00:00,2022-01-12T17:00:00
1,Hoàn Kiếm,VN,21.0285,105.8542,Asia/Ho_Chi_Minh,100,0.0,1019,1020,97,...,16.8,16.3,0.0,10.0,1.2,2.0,98,0,2022-01-13 01:00:00,2022-01-12T18:00:00
2,Hoàn Kiếm,VN,21.0285,105.8542,Asia/Ho_Chi_Minh,91,0.0,1019,1020,97,...,16.4,16.0,0.0,8.0,1.2,2.0,78,0,2022-01-13 02:00:00,2022-01-12T19:00:00
3,Hoàn Kiếm,VN,21.0285,105.8542,Asia/Ho_Chi_Minh,83,0.0,1019,1020,97,...,16.1,15.6,0.0,6.0,1.2,2.4,58,0,2022-01-13 03:00:00,2022-01-12T20:00:00
4,Hoàn Kiếm,VN,21.0285,105.8542,Asia/Ho_Chi_Minh,75,0.0,1018,1019,97,...,15.7,15.2,0.0,4.0,1.6,3.2,46,0,2022-01-13 04:00:00,2022-01-12T21:00:00


In [7]:
df_weather.to_csv('weather_data_1.csv', index=False)

In [8]:
df_weather = pd.read_csv('weather_data_1.csv')
# df_weather.info()

In [19]:
df = pd.read_csv('air_quality_data_complete.csv', index_col='Local Time', parse_dates=['Local Time'])
df_weather = pd.read_csv('weather_data_1.csv', index_col='Local Time', parse_dates=['Local Time'])


#### **IV. Hợp nhất hai khung dữ liệu và sắp xếp dữ liệu**

In [20]:

merged_df = pd.merge(df, df_weather, left_index=True, right_index=True)

merged_df.drop(columns=['City_y', 'Country code_y', 'timezone_y', 'UTC Time_y'], inplace=True)

utc_time_column = merged_df.pop('UTC Time_x')
merged_df.insert(0, 'UTC Time', utc_time_column)

merged_df = merged_df.rename(columns={'City_x': 'City', 'Country code_x': 'Country Code', 'timezone_x':'Timezone'})
merged_df

,UTC Time,City,Country Code,Timezone,AQI,CO,NO2,O3,PM10,PM25,...,Relative Humidity,Temperature,Apparent Temperature,Dew Point,UV Index,Visibility,Wind Speed,Wind Gust Speed,Wind Direction,Solar Radiation
Local Time,,,,,,,,,,,,,,,,,,,,,
2022-01-13 00:00:00,2022-01-12T17:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,180.0,381.3,16.7,86.7,109.3,77.67,...,93,17.0,17.0,15.9,0.0,10.0,1.60,2.4,106,0
2022-01-13 01:00:00,2022-01-12T18:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,181.0,389.8,17.0,87.0,111.0,79.00,...,97,16.8,16.8,16.3,0.0,10.0,1.20,2.0,98,0
2022-01-13 02:00:00,2022-01-12T19:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,178.0,385.0,16.0,87.3,107.3,76.33,...,97,16.4,16.4,16.0,0.0,8.0,1.20,2.0,78,0
2022-01-13 03:00:00,2022-01-12T20:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,174.0,380.1,15.0,87.7,103.7,73.67,...,97,16.1,16.1,15.6,0.0,6.0,1.20,2.4,58,0
2022-01-13 04:00:00,2022-01-12T21:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,171.0,375.3,14.0,88.0,100.0,71.00,...,97,15.7,15.7,15.2,0.0,4.0,1.60,3.2,46,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-31 19:00:00,2026-08-31T12:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,95.0,198.4,116.0,18.2,51.0,33.00,...,69,30.3,35.5,24.0,0.0,16.0,0.84,1.7,307,0
2026-08-31 20:00:00,2026-08-31T13:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,93.0,276.8,94.0,12.9,43.0,32.00,...,73,29.9,35.4,24.5,0.0,16.0,0.29,1.3,282,0
2026-08-31 21:00:00,2026-08-31T14:00:00,Hoàn Kiếm,VN,Asia/Ho_Chi_Minh,116.0,355.2,97.0,22.3,55.0,41.50,...,71,29.5,34.0,23.7,0.0,16.0,0.98,1.5,237,0


In [21]:

merged_df.shape

(40608, 27)

In [24]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 40608 entries, 2022-01-13 00:00:00 to 2026-08-31 23:00:00
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   UTC Time              40608 non-null  object 
 1   City                  40608 non-null  object 
 2   Country Code          40608 non-null  object 
 3   Timezone              40608 non-null  object 
 4   AQI                   40606 non-null  float64
 5   CO                    40534 non-null  float64
 6   NO2                   40534 non-null  float64
 7   O3                    40534 non-null  float64
 8   PM10                  40606 non-null  float64
 9   PM25                  40606 non-null  float64
 10  SO2                   40534 non-null  float64
 11  Lat                   40608 non-null  float64
 12  Lon                   40608 non-null  float64
 13  Clouds                40608 non-null  int64  
 14  Precipitation         40608 non-nul

In [26]:
merged_df.describe().T

,count,mean,std,min,25%,50%,75%,max
AQI,40606.0,122.743018,5.562143e+01,4.0000,82.0000,111.0000,155.0000,500.0000
CO,40534.0,634.734532,9.357018e+02,0.0000,126.6000,208.3000,791.0000,15956.2000
NO2,40534.0,30.742024,3.346805e+01,0.0000,12.0000,20.2000,36.5000,894.0000
O3,40534.0,50.969453,4.179904e+01,0.0000,21.0000,39.3000,69.3000,700.0000
PM10,40606.0,68.438743,6.628356e+01,1.0000,35.0000,52.5000,80.3000,1103.0000
PM25,40606.0,49.795819,3.848401e+01,1.0000,26.2200,39.0000,59.3300,457.0000
SO2,40534.0,53.764274,5.531692e+01,0.0000,15.0000,42.0000,78.0000,721.0000
Lat,40608.0,21.028500,3.552757e-15,21.0285,21.0285,21.0285,21.0285,21.0285
Lon,40608.0,105.854200,0.000000e+00,105.8542,105.8542,105.8542,105.8542,105.8542
Clouds,40608.0,64.427502,3.493224e+01,0.0000,37.0000,75.0000,100.0000,100.0000


In [27]:
merged_df.to_csv('air_quality_weather_data.csv', index=True)

In [28]:
# merged_df.rename(columns={'Local Time_x': 'Local Time'}, inplace=True)
# merged_df = merged_df[['Local Time', 'UTC Time', 'City', 'Country Code', 'Timezone', 'AQI', 'CO', 'NO2', 'O3', 'PM10', 'PM25', 'SO2',
#                        'Clouds', 'Precipitation', 'Pressure', 'Relative Humidity', 'Temperature', 'UV Index', 'Wind Speed']]

# merged_df.info()

In [ ]:
# Gộp các file lại csv
data22 = pd.read_csv('E:\Document\PROJECT\data\raw\data22.csv')
data2324 = pd.read_csv('E:\Document\PROJECT\data\raw\data2324.csv')


data2324 = data2324[data22.columns]
merge_data = pd.concat([data22, data2324])


merge_data.to_csv('E:\Document\PROJECT\data\raw\data2224.csv', index=False)

In [ ]:
merge_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26009 entries, 0 to 17543
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Local Time         26009 non-null  object 
 1   UTC Time           26009 non-null  object 
 2   City               26009 non-null  object 
 3   Country Code       26009 non-null  object 
 4   Timezone           26009 non-null  object 
 5   AQI                26009 non-null  int64  
 6   CO                 26009 non-null  float64
 7   NO2                26009 non-null  float64
 8   O3                 26009 non-null  float64
 9   PM10               26009 non-null  float64
 10  PM25               26009 non-null  float64
 11  SO2                26009 non-null  float64
 12  Clouds             26009 non-null  int64  
 13  Precipitation      26009 non-null  float64
 14  Pressure           26009 non-null  int64  
 15  Relative Humidity  26009 non-null  int64  
 16  Temperature        26009 no

In [ ]:
merge_data.interpolate(method='linear', limit_direction='forward', inplace=True)

C:\Users\hungd\AppData\Local\Temp\ipykernel_18700\1762461018.py:1: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  merge_data.interpolate(method='linear', limit_direction='forward', inplace=True)


In [ ]:

merged_df.to_csv('data/raw/data_test_2025.csv', index=False)